# Lab 02｜學生如何到校 Categorical Data

<a href="https://colab.research.google.com/github/johnnychao/statistics-in-context-bilingual/blob/main/labs/colab/lab-02-categorical-data.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

> Statistics in Context · Unit 1 · 原創合成資料 · 不評量 Python 語法


## Goal

情境：學校要調整晨間入口與自行車停放空間。學生主要如何到校？

- 建立 **frequency table（次數表）**與 **relative frequency（相對次數）**。
- 製作有情境、單位、標題與替代文字的 **bar chart（長條圖）**。
- 用比例回答問題，而不是只重述最高的長條。


## Setup

依序執行儲存格即可，不需要撰寫或背誦 Python。若想重新開始，請在 Colab 選擇 **Runtime → Restart session and run all**。

本 Lab 使用原創合成資料；所有代碼與數值均不對應真實學生。


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


In [ ]:
# 集中設定：一般情況只需修改這一格的參數。
DATA_RELATIVE_PATH = "data/public/morning_routine_survey.csv"
REPO_RAW_BASE_URL = "https://raw.githubusercontent.com/johnnychao/statistics-in-context-bilingual/main"

LOCAL_REPO_ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path("/content/statistics-in-context-bilingual"),
]


def load_repo_csv(relative_path):
    # 先找本機 repo，再讀 GitHub raw；失敗時提供繁中修復訊息。
    relative_path = Path(relative_path)
    for candidate_root in LOCAL_REPO_ROOT_CANDIDATES:
        candidate = candidate_root / relative_path
        if candidate.is_file():
            return pd.read_csv(candidate), str(candidate.resolve())

    remote_url = f"{REPO_RAW_BASE_URL}/{relative_path.as_posix()}"
    try:
        return pd.read_csv(remote_url), remote_url
    except Exception as exc:
        raise RuntimeError(
            "無法載入資料。請確認網路連線，或從 GitHub repo 根目錄執行此 Notebook。"
            f" 嘗試的遠端網址：{remote_url}。"
            " 若 repo 尚未發布，請先將 data/public 的 CSV 上傳至 main branch。"
        ) from exc


data, data_source = load_repo_csv(DATA_RELATIVE_PATH)
print(f"已載入 {len(data)} 筆資料｜Loaded {len(data)} rows")
print(f"來源 Source: {data_source}")


## Steps

### 1. 建立次數與比例表

先使用 `commute_mode`。之後可將參數改為 `ate_breakfast` 或 `arrival_status`。


In [ ]:
# ✏️ 修改任務：完成原分析後，換成另一個 categorical variable。
CATEGORY_VARIABLE = "commute_mode"

if CATEGORY_VARIABLE not in data.columns:
    raise ValueError(f"找不到欄位：{CATEGORY_VARIABLE}")

counts = data[CATEGORY_VARIABLE].value_counts(dropna=False)
category_table = pd.DataFrame({
    "frequency": counts,
    "relative_frequency": counts / counts.sum(),
})
display(category_table)
print(f"比例總和 Sum of proportions = {category_table['relative_frequency'].sum():.3f}")


### 2. 製作情境式長條圖

替代文字（alt text）讓無法看圖的讀者也能取得主要資訊。請先讀替代文字，再確認圖表是否真的支持它。


In [ ]:
FIGURE_ALT = (
    "Bar chart of synthetic students by primary commute mode; "
    "the bars compare counts for walk, bicycle, public transit, family car, and school shuttle."
)

plot_table = category_table.reset_index(names=CATEGORY_VARIABLE)
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=plot_table, x=CATEGORY_VARIABLE, y="frequency", color="#2A6F97", ax=ax)
ax.set_title("How synthetic students usually travel to school")
ax.set_xlabel("Primary commute mode（主要通勤方式）")
ax.set_ylabel("Number of synthetic students（人數）")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()
print(f"Alt text: {FIGURE_ALT}")


### 3. 用比例回答校務問題

修改 `FOCUS_CATEGORY`，取得一個可放進英文回答的具體數值。


In [ ]:
# ✏️ 修改任務：改成圖表中的另一個類別。
FOCUS_CATEGORY = "public_transit"

if FOCUS_CATEGORY not in category_table.index:
    raise ValueError(f"類別不存在：{FOCUS_CATEGORY}")

focus_count = int(category_table.loc[FOCUS_CATEGORY, "frequency"])
focus_proportion = float(category_table.loc[FOCUS_CATEGORY, "relative_frequency"])
print(f"{FOCUS_CATEGORY}: {focus_count} students ({focus_proportion:.1%})")


<details>
<summary><strong>AP English Response frame</strong></summary>

> In this synthetic survey, ____ out of 240 students (____%) reported ____. This suggests that the school should consider ____. The result describes this dataset and does not by itself represent every student at the school.

</details>

常見迷思：長條較高表示該類別人數較多，不表示其數值「比較大」或具有因果效果。


## Checks

檢查次數、比例與圖表使用的資料是否一致。


In [ ]:
assert int(category_table["frequency"].sum()) == len(data)
assert np.isclose(category_table["relative_frequency"].sum(), 1.0)
assert (category_table["relative_frequency"] >= 0).all()
print("✅ Checks passed：次數總和等於 240，比例總和等於 1。")


## Next Steps

Lab 03 將從 categorical data 轉向 quantitative data，使用通勤時間練習 shape、center、variability 與 unusual features。
